In [7]:
import numpy as np
import pandas as pd
from joblib import dump, load
import os
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error, 
    mean_squared_error,
    root_mean_squared_error, 
    mean_absolute_percentage_error,
    root_mean_squared_log_error,
    make_scorer
)
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor

### Data importation

In [8]:
df = pd.read_csv("../data/processed/bc_final.csv")
df.head(3)

,latitude,longitude,price,property-beds,property-baths,Acreage,Property Tax,Square Footage,Missing Acreage,Missing Property Tax,...,heat_pump,overhead,space_heater,Property Type_Condo,Property Type_Condo/Townhome,Property Type_Duplex,Property Type_Manufactured Home,Property Type_MultiFamily,Property Type_Single Family,Property Type_Townhome
0,49.821860,-119.480143,1298000.0,5.0,4.0,0.69,6995.0,4374.0,0,0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,49.138904,-122.654191,1399999.0,6.0,4.0,0.04,2585.0,2404.0,0,0,...,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,49.103726,-122.663125,399900.0,1.0,1.0,0.00,1474.0,632.0,1,0,...,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
X = df.drop(["price"], axis=1)
Y = df["price"]

#### train test split

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, shuffle=True
)

### Search n°1

In [6]:
ada_search_space = {
    "n_estimators" : Integer(20, 600),
    "learning_rate" : Categorical([0.5, 1, 5]),

    "estimator__max_depth": Integer(3, 15),
    "estimator__min_samples_split" : Integer(30, 100),
    "estimator__min_samples_leaf" : Integer(20, 100),
    "estimator__max_features": Categorical([None, "sqrt", "log2"])
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_rmsle" : make_scorer(root_mean_squared_log_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}

#### BayesSearchCV

In [7]:
ada_search = BayesSearchCV(
    estimator = AdaBoostRegressor(
        estimator = DecisionTreeRegressor(random_state=42),
        random_state=42
    ),
    search_spaces=ada_search_space,
    scoring = scoring["neg_mse"],
    n_iter = 50,
    cv=KFold(3),
    n_jobs=3,
    error_score="raise",
    verbose=2
)

In [9]:
if os.path.isfile("../artifacts/ada_search.pkl"):
    print("The object already exists")
else : 
    ada_search.fit(X_train, Y_train)
    dump(ada_search, "../artifacts/ada_search.pkl")
    print("The object has been successfully saved")

The object already exists


In [10]:
ada_search = load("../artifacts/ada_search.pkl")
ada_search.best_params_

OrderedDict([('estimator__max_depth', 15),
             ('estimator__max_features', None),
             ('estimator__min_samples_leaf', 20),
             ('estimator__min_samples_split', 30),
             ('learning_rate', 1),
             ('n_estimators', 20)])

In [12]:
best_model_ada = ada_search.best_estimator_

y_test_pred = best_model_ada.predict(X_test)
y_train_pred = best_model_ada.predict(X_train)

In [13]:
print("R²:", r2_score(Y_test, y_test_pred))
print("R²:", r2_score(Y_train, y_train_pred))

print("MAE:", mean_absolute_error(Y_test, y_test_pred))
print("MAE:", mean_absolute_error(Y_train, y_train_pred))

print("RMSE:", root_mean_squared_error(Y_test, y_test_pred))
print("RMSE:", root_mean_squared_error(Y_train, y_train_pred))

print("MAPE:", mean_absolute_percentage_error(Y_test, y_test_pred))
print("MAPE:", mean_absolute_percentage_error(Y_train, y_train_pred))

print("RMSLE", root_mean_squared_log_error(Y_test, y_test_pred))
print("RMSLE", root_mean_squared_log_error(Y_train, y_train_pred))

R²: 0.8383826081433771
R²: 0.9666429200671184
MAE: 304804.9408324189
MAE: 209041.5246368083
RMSE: 699205.9824795482
RMSE: 334739.0647919193
MAPE: 0.19821025430397385
MAPE: 0.16417924643891263
RMSLE 0.2587383077041086
RMSLE 0.2143856541302239


We'll try to reduce the learning_rate, in order to make the prediction converge slowly to the real price, without aggressivity.

In [15]:
df_error = pd.DataFrame({
    "y_true": Y_test,
    "y_pred": y_test_pred
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_22303/3528320552.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_22303/3528320552.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(64899.999, 638000.0]      0.327886
(638000.0, 889000.0]       0.137076
(889000.0, 1327400.0]      0.143523
(1327400.0, 2043132.8]     0.181408
(2043132.8, 28888000.0]    0.200895
dtype: float64

The mean average percentage error on each price range shows us a bigger error on the cheapest houses of the dataset.  
We'll try to re-train our model with the RMSLE loss function which will penalize the model more for the smallest prices.

### Search n°2

In [17]:
ada_search2 = BayesSearchCV(
    estimator = AdaBoostRegressor(
        estimator = DecisionTreeRegressor(random_state=42),
        random_state=42
    ),
    search_spaces=ada_search_space,
    scoring = scoring["neg_rmsle"],
    n_iter = 35,
    cv=KFold(3),
    n_jobs=3,
    error_score="raise",
    verbose=2
)

In [19]:
if os.path.isfile("../artifacts/ada_search2.pkl"):
    print("The object already exists")
else : 
    ada_search2.fit(X_train, Y_train)
    dump(ada_search2, "../artifacts/ada_search2.pkl")
    print("The object has been successfully saved")

The object already exists


In [20]:
ada_search2 = load("../artifacts/ada_search2.pkl")
ada_search2.best_params_

OrderedDict([('estimator__max_depth', 15),
             ('estimator__max_features', None),
             ('estimator__min_samples_leaf', 20),
             ('estimator__min_samples_split', 30),
             ('learning_rate', 0.5),
             ('n_estimators', 24)])

In [22]:
best_model_ada2 = ada_search2.best_estimator_

y_test_pred2 = best_model_ada2.predict(X_test)
y_train_pred2 = best_model_ada2.predict(X_train)

In [23]:
print("R²:", r2_score(Y_test, y_test_pred2))
print("R²:", r2_score(Y_train, y_train_pred2))

print("MAE:", mean_absolute_error(Y_test, y_test_pred2))
print("MAE:", mean_absolute_error(Y_train, y_train_pred2))

print("RMSE:", root_mean_squared_error(Y_test, y_test_pred2))
print("RMSE:", root_mean_squared_error(Y_train, y_train_pred2))

print("MAPE:", mean_absolute_percentage_error(Y_test, y_test_pred2))
print("MAPE:", mean_absolute_percentage_error(Y_train, y_train_pred2))

print("RMSLE", root_mean_squared_log_error(Y_test, y_test_pred2))
print("RMSLE", root_mean_squared_log_error(Y_train, y_train_pred2))

R²: 0.8264176282909568
R²: 0.9588070114138955
MAE: 299810.20683245437
MAE: 213822.53590456062
RMSE: 724625.9746536954
RMSE: 371983.80834581883
MAPE: 0.18680719831295714
MAPE: 0.1547531509495451
RMSLE 0.249839044853165
RMSLE 0.20632490284854643


In [25]:
df_error = pd.DataFrame({
    "y_true": Y_test,
    "y_pred": y_test_pred2
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_22303/512691629.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_22303/512691629.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(64899.999, 638000.0]      0.289378
(638000.0, 889000.0]       0.129540
(889000.0, 1327400.0]      0.139520
(1327400.0, 2043132.8]     0.173794
(2043132.8, 28888000.0]    0.201594
dtype: float64

The MAPE scores on each price range is lower than the previous model.  
The RMSLE as a scoring method gives us better results than the previous search. It's due to the logarithm applied to $y$ and $\hat{y}$. It penalizes the model for the mistakes made on the smallest values.

### Search n°3

#### This time, we'll transform the Y vector target to a log-Y target vector in order to reduce the errors for the smallest prices of our dataset.  

Indeed, by applying the logarithm to the `Price` feature, the model penalizes much more the errors for the smallest price values than the biggest ones.  
We do that in order to give more importance to the errors on the smallest prices, by removing the dominance of large prices. We change the "space" of the error space thanks to the form of the logartihm function that "flattens" the large values.

In [10]:
Y_log = np.log1p(Y)

#### New train test split

In [11]:
X_train2, X_test2, Y_train2, Y_test2 = train_test_split(
    X, Y_log, test_size=0.20, random_state=42, shuffle=True
)

In [29]:
ada_search_space2 = {
    "n_estimators" : Integer(30, 300),
    "learning_rate" : Real(0.01, 0.5),

    "estimator__max_depth": Integer(3, 15),
    "estimator__min_samples_split" : Integer(30, 100),
    "estimator__min_samples_leaf" : Integer(20, 100),
    "estimator__max_features": Categorical([None, "sqrt", "log2"])
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_rmsle" : make_scorer(root_mean_squared_log_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}

ada_search3 = BayesSearchCV(
    estimator = AdaBoostRegressor(
        estimator = DecisionTreeRegressor(random_state=42),
        random_state=42
    ),
    search_spaces=ada_search_space2,
    scoring = scoring["neg_mse"],
    n_iter = 50,
    cv=KFold(3),
    n_jobs=3,
    error_score="raise",
    verbose=2
)

In [31]:
if os.path.isfile("../artifacts/ada_search3.pkl"):
    print("The object already exists")
else : 
    ada_search3.fit(X_train2, Y_train2)
    dump(ada_search3, "../artifacts/ada_search3.pkl")
    print("The object has been successfully saved")

The object already exists


In [3]:
ada_search3 = load("../artifacts/ada_search3.pkl")
ada_search3.best_params_

OrderedDict([('estimator__max_depth', 15),
             ('estimator__max_features', None),
             ('estimator__min_samples_leaf', 20),
             ('estimator__min_samples_split', 30),
             ('learning_rate', 0.5),
             ('n_estimators', 300)])

In [12]:
best_model_ada3 = ada_search3.best_estimator_

y_pred_log = best_model_ada3.predict(X_test2)
y_test_pred3 = np.expm1(y_pred_log)

y_train_pred_log = best_model_ada3.predict(X_train2)
y_train_pred3 = np.expm1(y_train_pred_log)

In [13]:
print("R²:", r2_score(np.expm1(Y_test2), y_test_pred3))
print("R²:", r2_score(np.expm1(Y_train2), y_train_pred3))

print("MAE:", mean_absolute_error(np.expm1(Y_test2), y_test_pred3))
print("MAE:", mean_absolute_error(np.expm1(Y_train2), y_train_pred3))

print("RMSE:", root_mean_squared_error(np.expm1(Y_test2), y_test_pred3))
print("RMSE:", root_mean_squared_error(np.expm1(Y_train2), y_train_pred3))

print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_test2), y_test_pred3))
print("MAPE:", mean_absolute_percentage_error(np.expm1(Y_train2), y_train_pred3))

print("RMSLE", root_mean_squared_log_error(np.expm1(Y_test2), y_test_pred3))
print("RMSLE", root_mean_squared_log_error(np.expm1(Y_train2), y_train_pred3))

R²: 0.8189949519776325
R²: 0.9458579944350268
MAE: 291623.389946489
MAE: 206970.48776838958
RMSE: 739956.9132376957
RMSE: 426461.2300358837
MAPE: 0.16423578312489984
MAPE: 0.11937094369129066
RMSLE 0.2257359377033309
RMSLE 0.14235975166741627


In [14]:
df_error = pd.DataFrame({
    "y_true": np.expm1(Y_test2),
    "y_pred": y_test_pred3
})

#this creates a new column with "q" different price ranges. We can know where to situtate each row.
df_error["price_bin"] = pd.qcut(df_error["y_true"], q=5)  

# We calculate the Mean Average Percentage Error for each price range.
df_error.groupby("price_bin").apply(
    lambda x: np.mean(np.abs((x.y_true - x.y_pred) / x.y_true))
)

/tmp/ipykernel_63767/1809920531.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_error.groupby("price_bin").apply(
/tmp/ipykernel_63767/1809920531.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_error.groupby("price_bin").apply(


price_bin
(64899.999, 638000.0]      0.200577
(638000.0, 889000.0]       0.123240
(889000.0, 1327400.0]      0.131208
(1327400.0, 2043132.8]     0.169053
(2043132.8, 28888000.0]    0.196994
dtype: float64

#### Conclusion

AdaBoostRegressor with a DecisionTreeRegressor performs better than a Random Forest, without overfiting too.  
The mean average percentage error for each range of prices is less or equal to 20%, which is pretty good for a housing price estimator trained on a dataset with a large range of prices (from **64'899\\$** to **58'000'000\\$**).  
Even if the model gives us good estimates, we'll train a GradientBoostingRegressor combined to a DecisionTreeRegressor, hoping to take advantage of its robustness and obtain better predictions.

In [15]:
if os.path.isfile("../artifacts/ada_best_model.pkl"):
    print("The object already exists")
else : 
    dump(ada_search3.best_estimator_, "../artifacts/ada_best_model.pkl")
    print("The object has been successfully saved")

The object has been successfully saved
